# Notebook 02 — Analytics: DuckDB поверх Parquet

**Что делаем:** запускаем аналитические запросы прямо поверх Parquet-файлов без загрузки в БД.

Из семинара: DuckDB — это **in-process OLAP**. Никаких серверов, никакой инфраструктуры.
> `SELECT * FROM 'data/*.parquet'` — работает из коробки.

**Предварительно:** убедитесь, что в `data/` есть файлы:
```bash
python setup/generate_data.py   # создаёт data/raw_orders.parquet
```

In [1]:
import duckdb
import pandas as pd
import time

con = duckdb.connect()   # :memory: — ничего не сохраняется на диск
print("DuckDB", duckdb.__version__)

DuckDB 2.0.0


## 1. Читаем Parquet напрямую

In [2]:
# DuckDB читает файл без загрузки в память целиком
con.execute("SELECT * FROM '../data/raw_orders.parquet' LIMIT 5").df()

,order_id,customer_id,product,category,quantity,unit_price,discount,amount,status,order_date,created_at
0,1,90,Пылесос,Home,3,9000.0,0.00,27000.0,pending,2023-05-17,2023-05-17 11:50:00
1,2,106,Смартфон,Electronics,1,35000.0,0.00,35000.0,shipped,2023-08-24,2023-08-24 17:19:00
2,3,111,Куртка,Clothing,1,6500.0,0.00,6500.0,pending,2023-07-24,2023-07-24 16:17:00
3,4,15,Пылесос,Home,3,9000.0,0.15,22950.0,cancelled,2023-07-24,2023-07-24 16:01:00
4,5,56,Кофемашина,Home,3,12000.0,0.05,34200.0,shipped,2023-01-20,2023-01-20 09:38:00


In [3]:
# Сколько строк и какие статусы?
con.execute("""
    SELECT status, count(*) AS cnt, round(avg(amount), 2) AS avg_amount
    FROM '../data/raw_orders.parquet'
    GROUP BY status
    ORDER BY cnt DESC
""").df()

,status,cnt,avg_amount
0,shipped,837,22667.79
1,pending,373,21969.34
2,cancelled,170,20922.51
3,returned,120,23917.85


## 2. Скорость: DuckDB vs Postgres

DuckDB читает **только нужные колонки** из Parquet (column pruning) — это ключевое преимущество для аналитики.

In [4]:
# Смотрим план запроса — видим PARQUET_SCAN
con.execute("""
    EXPLAIN
    SELECT date_trunc('month', order_date) AS month, sum(amount)
    FROM '../data/raw_orders.parquet'
    WHERE status = 'shipped'
    GROUP BY 1
""").df()

,explain_key,explain_value
0,physical_plan,╭─ Hash Group By ─────────────╮\n│ Groups: #0 ...


## ✏️ Задание 1: выручка по месяцам

Напишите запрос, который считает **помесячную выручку** (только `status = 'shipped'`).

Ожидаемый результат:
| month | total_revenue | order_count |
|-------|--------------|-------------|
| 2023-01-01 | ... | ... |

In [5]:
result = con.execute("""
    SELECT
        date_trunc('month', order_date)::date AS month,
        sum(amount)                           AS total_revenue,
        count(*)                              AS order_count
    FROM '../data/raw_orders.parquet'
    WHERE status = 'shipped'
    GROUP BY 1
    ORDER BY 1
""").df()
result

,month,total_revenue,order_count
0,2023-01-01,1156920.0,58
1,2023-02-01,1904782.5,71
2,2023-03-01,1406982.5,89
3,2023-04-01,2215980.0,78
4,2023-05-01,1303980.0,66
5,2023-06-01,1580110.0,71
6,2023-07-01,2439815.0,85
7,2023-08-01,1418660.0,60
8,2023-09-01,1543412.5,64
9,2023-10-01,1466047.5,62


<details>
<summary>💡 Подсказка</summary>

```sql
SELECT
    date_trunc('month', order_date)::date AS month,
    sum(amount)                           AS total_revenue,
    count(*)                              AS order_count
FROM '../data/raw_orders.parquet'
WHERE status = 'shipped'
GROUP BY 1
ORDER BY 1
```
</details>

## 3. JOIN двух Parquet файлов

In [6]:
# DuckDB умеет JOIN-ить разные файлы
con.execute("""
    SELECT
        c.segment,
        c.city,
        count(DISTINCT o.order_id)  AS orders,
        round(sum(o.amount), 0)     AS revenue
    FROM '../data/raw_orders.parquet'    AS o
    JOIN '../data/raw_customers.parquet' AS c USING (customer_id)
    WHERE o.status = 'shipped'
    GROUP BY c.segment, c.city
    ORDER BY revenue DESC
    LIMIT 10
""").df()

,segment,city,orders,revenue
0,B2C,Екатеринбург,145,3616948.0
1,B2C,Москва,142,2995768.0
2,B2C,Санкт-Петербург,103,2754980.0
3,B2C,Нижний Новгород,59,1693320.0
4,B2C,Самара,86,1623215.0
5,B2C,Новосибирск,59,1348595.0
6,B2C,Челябинск,69,1266090.0
7,B2C,Казань,55,1138075.0
8,B2B,Казань,38,1053400.0
9,B2B,Москва,19,414795.0


## ✏️ Задание 2: топ категорий по выручке

Найдите **топ-3 категории товаров** по суммарной выручке. Для каждой категории покажите:  
`category | total_revenue | avg_discount | order_count`

In [7]:
result2 = con.execute("""
    SELECT
        category,
        round(sum(amount), 0)    AS total_revenue,
        round(avg(discount), 3)  AS avg_discount,
        count(*)                 AS order_count
    FROM '../data/raw_orders.parquet'
    WHERE status = 'shipped'
    GROUP BY category
    ORDER BY total_revenue DESC
    LIMIT 3
""").df()
result2

,category,total_revenue,avg_discount,order_count
0,Electronics,10401350.0,0.056,160
1,Sports,4216045.0,0.060,187
2,Clothing,2090615.0,0.057,175


## 4. Сохраняем снапшот (витрину)

Из семинара: **снапшот = проекция лога в момент T**. Храним в Parquet для производительности потребителей.

In [8]:
# Сохраняем витрину выручки по месяцам
con.execute("""
    COPY (
        SELECT
            date_trunc('month', order_date)::date AS month,
            c.segment,
            count(*)                              AS total_orders,
            round(sum(o.amount), 2)               AS total_revenue,
            round(avg(o.amount), 2)               AS avg_order_value
        FROM '../data/raw_orders.parquet'    AS o
        JOIN '../data/raw_customers.parquet' AS c USING (customer_id)
        WHERE o.status = 'shipped'
        GROUP BY 1, 2
        ORDER BY 1, 2
    ) TO '../data/mart_revenue.parquet' (FORMAT PARQUET, COMPRESSION SNAPPY)
""")
print("✓ data/mart_revenue.parquet создан")

# Читаем обратно
con.execute("SELECT * FROM '../data/mart_revenue.parquet' LIMIT 6").df()

✓ data/mart_revenue.parquet создан


,month,segment,total_orders,total_revenue,avg_order_value
0,2023-01-01,B2B,6,120045.0,20007.50
1,2023-01-01,B2C,52,1036875.0,19939.90
2,2023-02-01,B2B,11,151815.0,13801.36
3,2023-02-01,B2C,60,1752967.5,29216.13
4,2023-03-01,B2B,8,98570.0,12321.25
5,2023-03-01,B2C,81,1308412.5,16153.24


## 5. DuckDB ↔ MinIO (S3)

В production данные хранятся в S3/MinIO. DuckDB подключается прозрачно:

In [9]:
# Если у вас запущен MinIO — раскомментируйте:
# con.execute("""
#     INSTALL httpfs; LOAD httpfs;
#     SET s3_endpoint         = 'localhost:9000';
#     SET s3_access_key_id    = 'minioadmin';
#     SET s3_secret_access_key= 'minioadmin';
#     SET s3_use_ssl          = false;
# """)
# df = con.execute("SELECT * FROM 's3://datalake/orders/*.parquet'").df()
print("MinIO секция — для продвинутой части. Сейчас используем локальные файлы.")

MinIO секция — для продвинутой части. Сейчас используем локальные файлы.


---
**Готово!** Теперь у нас есть:
- `data/raw_orders.parquet` — исходный лог
- `data/mart_revenue.parquet` — снапшот (витрина)

Переходим к Notebook 03 — создаём те же трансформации через **dbt**.